In [108]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
books = []

for page in range(1, 6):  # 5 páginas ≈ 500 libros
    url = f"https://openlibrary.org/search.json?q=psychology&page={page}"
    
    response = requests.get(url)
    data = response.json()

    for book in data["docs"]:
        books.append({
            "title": book.get("title"),
            "author": ", ".join(book.get("author_name", [])) if book.get("author_name") else None,
            "year": book.get("first_publish_year"),
            "cover_id": book.get("cover_i"),
            "work_key": book.get("key")
        })

    print(f"Page {page} done")

    time.sleep(1)  # importante para no saturar API

api_df = pd.DataFrame(books)

print(len(api_df))
api_df.head()

Page 1 done
Page 2 done
Page 3 done
Page 4 done
Page 5 done
500


,title,author,year,cover_id,work_key
0,Dark Psychology,Modern Psychology Publishing,2017.0,13262384.0,/works/OL30150379W
1,Dream Psychology,Sigmund Freud,2017.0,13180633.0,/works/OL29743214W
2,Educational psychology,"Anita Woolfolk Hoy, Philip H. Winne, Nancy E. ...",1987.0,137962.0,/works/OL2651233W
3,Flow,Mihaly Csikszentmihalyi,1990.0,11041932.0,/works/OL278571W
4,Psychology,Wayne Weiten,1989.0,363719.0,/works/OL90044W


In [15]:
def get_book_details(work_key):

    try:
        url = f"https://openlibrary.org{work_key}.json"

        response = requests.get(url, timeout=10)  # 🔥 evita que se cuelgue

        data = response.json()

        # DESCRIPTION
        description = data.get("description", "")
        if isinstance(description, dict):
            description = description.get("value", "")

        # SUBJECTS
        subjects = data.get("subjects", [])

        return {
            "description": description,
            "subjects": ", ".join(subjects)
        }

    except:
        return {
            "description": "",
            "subjects": ""
        }

In [16]:
for idx, row in api_df.iterrows():

    details = get_book_details(row["work_key"])

    enriched_books.append({
        "title": row["title"],
        "author": row["author"],
        "year": row["year"],
        "cover_id": row["cover_id"],
        "work_key": row["work_key"],
        "description": details["description"],
        "subjects": details["subjects"]
    })

    if idx % 10 == 0:
        print(f"Processed: {idx}")

    time.sleep(0.5)  # más rápido pero seguro

Processed: 0
Processed: 10
Processed: 20
Processed: 30
Processed: 40
Processed: 50
Processed: 60
Processed: 70
Processed: 80
Processed: 90
Processed: 100
Processed: 110
Processed: 120
Processed: 130
Processed: 140
Processed: 150
Processed: 160
Processed: 170
Processed: 180
Processed: 190
Processed: 200
Processed: 210
Processed: 220
Processed: 230
Processed: 240
Processed: 250
Processed: 260
Processed: 270
Processed: 280
Processed: 290
Processed: 300
Processed: 310
Processed: 320
Processed: 330
Processed: 340
Processed: 350
Processed: 360
Processed: 370
Processed: 380
Processed: 390
Processed: 400
Processed: 410
Processed: 420
Processed: 430
Processed: 440
Processed: 450
Processed: 460
Processed: 470
Processed: 480
Processed: 490


In [17]:
api_df.isnull().sum()

title        0
author       2
year         3
cover_id    29
work_key     0
dtype: int64

In [18]:
api_df.head()

,title,author,year,cover_id,work_key
0,Dark Psychology,Modern Psychology Publishing,2017.0,13262384.0,/works/OL30150379W
1,Dream Psychology,Sigmund Freud,2017.0,13180633.0,/works/OL29743214W
2,Educational psychology,"Anita Woolfolk Hoy, Philip H. Winne, Nancy E. ...",1987.0,137962.0,/works/OL2651233W
3,Flow,Mihaly Csikszentmihalyi,1990.0,11041932.0,/works/OL278571W
4,Psychology,Wayne Weiten,1989.0,363719.0,/works/OL90044W


In [19]:
api_df["author"] = api_df["author"].fillna("Unknown")
api_df["year"] = api_df["year"].fillna(0)
api_df["cover_id"] = api_df["cover_id"].fillna(0)

In [20]:
api_df["cover_url"] = api_df["cover_id"].apply(
    lambda x: f"https://covers.openlibrary.org/b/id/{int(x)}-M.jpg"
    if x != 0 else None
)

In [21]:
import requests
import time

def get_details(work_key):
    try:
        url = f"https://openlibrary.org{work_key}.json"
        r = requests.get(url, timeout=10)
        data = r.json()

        description = data.get("description", "")
        if isinstance(description, dict):
            description = description.get("value", "")

        subjects = data.get("subjects", [])

        return description, ", ".join(subjects)

    except:
        return "", ""

In [22]:
descriptions = []
subjects_list = []

for i, row in api_df.iterrows():

    desc, subs = get_details(row["work_key"])

    descriptions.append(desc)
    subjects_list.append(subs)

    if i % 10 == 0:
        print(f"Processed: {i}")

    time.sleep(0.5)

Processed: 0
Processed: 10
Processed: 20
Processed: 30
Processed: 40
Processed: 50
Processed: 60
Processed: 70
Processed: 80
Processed: 90
Processed: 100
Processed: 110
Processed: 120
Processed: 130
Processed: 140
Processed: 150
Processed: 160
Processed: 170
Processed: 180
Processed: 190
Processed: 200
Processed: 210
Processed: 220
Processed: 230
Processed: 240
Processed: 250
Processed: 260
Processed: 270
Processed: 280
Processed: 290
Processed: 300
Processed: 310
Processed: 320
Processed: 330
Processed: 340
Processed: 350
Processed: 360
Processed: 370
Processed: 380
Processed: 390
Processed: 400
Processed: 410
Processed: 420
Processed: 430
Processed: 440
Processed: 450
Processed: 460
Processed: 470
Processed: 480
Processed: 490


In [23]:
api_df["description"] = descriptions
api_df["subjects"] = subjects_list

In [24]:
api_df.info()
api_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        500 non-null    object 
 1   author       500 non-null    object 
 2   year         500 non-null    float64
 3   cover_id     500 non-null    float64
 4   work_key     500 non-null    object 
 5   cover_url    471 non-null    object 
 6   description  500 non-null    object 
 7   subjects     500 non-null    object 
dtypes: float64(2), object(6)
memory usage: 31.4+ KB


,title,author,year,cover_id,work_key,cover_url,description,subjects
0,Dark Psychology,Modern Psychology Publishing,2017.0,13262384.0,/works/OL30150379W,https://covers.openlibrary.org/b/id/13262384-M...,Have you ever felt manipulated or taken advant...,"Psychology, Human behavior, Manipulation, Pers..."
1,Dream Psychology,Sigmund Freud,2017.0,13180633.0,/works/OL29743214W,https://covers.openlibrary.org/b/id/13180633-M...,The Interpretation of Dreams is a book by Sigm...,"Psychology, psychoanalysis, dreams, dream inte..."
2,Educational psychology,"Anita Woolfolk Hoy, Philip H. Winne, Nancy E. ...",1987.0,137962.0,/works/OL2651233W,https://covers.openlibrary.org/b/id/137962-M.jpg,A text to provide beginning teachers with the ...,"Educational psychology, Textbooks, Manuels, Ps..."
3,Flow,Mihaly Csikszentmihalyi,1990.0,11041932.0,/works/OL278571W,https://covers.openlibrary.org/b/id/11041932-M...,Psychologist Mihaly Csikszentmihalyi's famous ...,"Nonfiction, Psychology, Lebensfreude, Flow-Erl..."
4,Psychology,Wayne Weiten,1989.0,363719.0,/works/OL90044W,https://covers.openlibrary.org/b/id/363719-M.jpg,"Weiten's PSYCHOLOGY: THEMES AND VARIATIONS, 11...","Psychology, Textbooks, Psychology textbooks, P..."


In [25]:
api_df.isnull().sum()

title           0
author          0
year            0
cover_id        0
work_key        0
cover_url      29
description     0
subjects        0
dtype: int64

In [26]:
api_df["description"] = api_df["description"].fillna("")
api_df["subjects"] = api_df["subjects"].fillna("")

In [27]:
api_df["content"] = (
    api_df["title"] + " " +
    api_df["author"] + " " +
    api_df["description"] + " " +
    api_df["subjects"]
)

In [28]:
api_df.to_csv("../data/processed/books_enriched_final.csv", index=False)

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# limpiar
api_df["content"] = api_df["content"].fillna("").astype(str)

# TF-IDF
vectorizer = TfidfVectorizer(stop_words="english")
matrix = vectorizer.fit_transform(api_df["content"])

# similitud
similarity = cosine_similarity(matrix)

print(similarity.shape)

(500, 500)


In [34]:
import numpy as np

def recommend(title, top_n=10):

    # encontrar índice del libro
    idx = api_df[api_df["title"] == title].index[0]

    # scores de similitud
    scores = list(enumerate(similarity[idx]))

    # ordenar por similitud (descendente)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # quitar el mismo libro
    scores = scores[1:top_n+1]

    # obtener resultados
    recommendations = []
    for i, score in scores:
        recommendations.append(api_df.iloc[i])

    return pd.DataFrame(recommendations)

In [35]:
recommend("Flow")

,title,author,year,cover_id,work_key,cover_url,description,subjects,content
374,Finding flow,Mihaly Csikszentmihalyi,1997.0,298856.0,/works/OL506559W,https://covers.openlibrary.org/b/id/298856-M.jpg,Based on a far-reaching study of thousands of ...,"Conduct of life, Happiness, Gedrag, Morale pra...",Finding flow Mihaly Csikszentmihalyi Based on ...
345,Applied positive psychology,"Stewart I. Donaldson, Jeanne Nakamura, Mihaly ...",2011.0,13586795.0,/works/OL16162280W,https://covers.openlibrary.org/b/id/13586795-M...,,"Applied Psychology, Psychology, applied, Perso...",Applied positive psychology Stewart I. Donalds...
351,Creativity,Mihaly Csikszentmihalyi,1996.0,3845086.0,/works/OL506563W,https://covers.openlibrary.org/b/id/3845086-M.jpg,Creativity is about capturing those moments th...,"Creative ability, Creative thinking, Nonfictio...",Creativity Mihaly Csikszentmihalyi Creativity ...
349,Authentic Happiness,Martin Elias Pete Seligman,2002.0,472177.0,/works/OL8004627W,https://covers.openlibrary.org/b/id/472177-M.jpg,,"Positive psychology, Happiness, Cognition & co...",Authentic Happiness Martin Elias Pete Seligman...
263,The happiness advantage,Shawn Achor,2010.0,6938464.0,/works/OL15674042W,https://covers.openlibrary.org/b/id/6938464-M.jpg,Recent discoveries in the field of positive ps...,"Work, Positive psychology, Psychological aspec...",The happiness advantage Shawn Achor Recent dis...
356,The psychology of happiness,Michael Argyle,1987.0,265917.0,/works/OL3903941W,https://covers.openlibrary.org/b/id/265917-M.jpg,,"Happiness, Psychologie, Happiness [MESH], Bonh...",The psychology of happiness Michael Argyle Ha...
99,Experience psychology,"Laura King, Laura A. King",2010.0,12644846.0,/works/OL9761658W,https://covers.openlibrary.org/b/id/12644846-M...,,Psychology,"Experience psychology Laura King, Laura A. Kin..."
321,Positive Psychology,Rona Hart,2020.0,13315352.0,/works/OL20754313W,https://covers.openlibrary.org/b/id/13315352-M...,,"Positive psychology, Psychologie positive, PSY...",Positive Psychology Rona Hart Positive psycho...
495,Positive psychology,William C. Compton,2012.0,8124000.0,/works/OL16626473W,https://covers.openlibrary.org/b/id/8124000-M.jpg,"""Topically organized, Positive Psychology: The...","Positive psychology, Self-actualization (psych...","Positive psychology William C. Compton ""Topica..."
496,Psychology for living,"Karen Grover Duffy, Eastwood Atwater",2001.0,1107256.0,/works/OL3519672W,https://covers.openlibrary.org/b/id/1107256-M.jpg,,"Textbooks, Conduct of life, Adjustment (psycho...","Psychology for living Karen Grover Duffy, East..."


In [41]:
import requests
import pandas as pd
import time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [42]:
session = requests.Session()

retry = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[500, 502, 503, 504]
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("https://", adapter)
session.mount("http://", adapter)

In [44]:
def fetch_books(query, limit=100, offset=0):

    url = "https://openlibrary.org/search.json"

    params = {
        "q": query,
        "limit": limit,
        "offset": offset
    }

    try:
        response = session.get(url, params=params, timeout=10)
        data = response.json()
        return data.get("docs", [])

    except Exception as e:
        print(f"Error en query={query}, offset={offset}: {e}")
        return []

In [45]:
def scrape_books(total=500):

    all_books = []
    seen = set()

    queries = [
        "psychology",
        "philosophy",
        "business",
        "self help",
        "fiction",
        "productivity"
    ]

    offset = 0

    while len(all_books) < total:

        for q in queries:

            books = fetch_books(q, limit=50, offset=offset)

            for b in books:

                key = b.get("key")
                if not key or key in seen:
                    continue

                seen.add(key)

                all_books.append({
                    "title": b.get("title"),
                    "author": ", ".join(b.get("author_name", [])) if b.get("author_name") else None,
                    "year": b.get("first_publish_year"),
                    "cover_id": b.get("cover_i"),
                    "work_key": key
                })

                if len(all_books) >= total:
                    break

            if len(all_books) >= total:
                break

        offset += 50

        print("Processed:", len(all_books))

        time.sleep(1.2)  # IMPORTANTE

    return pd.DataFrame(all_books[:total])

In [46]:
api_df = scrape_books(500)

print(len(api_df))
api_df.head()

Processed: 299
Processed: 500
500


,title,author,year,cover_id,work_key
0,Dark Psychology,Modern Psychology Publishing,2017.0,13262384.0,/works/OL30150379W
1,Dream Psychology,Sigmund Freud,2017.0,13180633.0,/works/OL29743214W
2,Educational psychology,"Anita Woolfolk Hoy, Philip H. Winne, Nancy E. ...",1987.0,137962.0,/works/OL2651233W
3,Flow,Mihaly Csikszentmihalyi,1990.0,11041932.0,/works/OL278571W
4,Psychology,Wayne Weiten,1989.0,363719.0,/works/OL90044W


In [48]:
import pandas as pd

df = pd.read_csv("../data/processed/books_enriched.csv")

print(len(df))
df.head()

100


,title,price,rating_text,link,author,description,genres
0,A Light in the Attic,£51.77,Three,a-light-in-the-attic_1000/index.html,Shel Silverstein,NaN,NaN
1,Tipping the Velvet,£53.74,One,tipping-the-velvet_999/index.html,Sarah Waters,NaN,NaN
2,Soumission,£50.10,One,soumission_998/index.html,Michel Houellebecq,NaN,NaN
3,Sharp Objects,£47.82,Four,sharp-objects_997/index.html,Gillian Flynn,NaN,NaN
4,Sapiens: A Brief History of Humankind,£54.23,Five,sapiens-a-brief-history-of-humankind_996/index...,BookNation,NaN,NaN


In [49]:
df_api = api_df.copy()
df_scrap = df.copy()

df_api = df_api.drop_duplicates(subset=["work_key"])
df_scrap = df_scrap.drop_duplicates(subset=["title"])

df_api["source"] = "openlibrary"
df_scrap["source"] = "scraping"

final_df = pd.concat([df_api, df_scrap], ignore_index=True)

print(len(final_df))
final_df.head()

600


,title,author,year,cover_id,work_key,source,price,rating_text,link,description,genres
0,Dark Psychology,Modern Psychology Publishing,2017.0,13262384.0,/works/OL30150379W,openlibrary,NaN,NaN,NaN,NaN,NaN
1,Dream Psychology,Sigmund Freud,2017.0,13180633.0,/works/OL29743214W,openlibrary,NaN,NaN,NaN,NaN,NaN
2,Educational psychology,"Anita Woolfolk Hoy, Philip H. Winne, Nancy E. ...",1987.0,137962.0,/works/OL2651233W,openlibrary,NaN,NaN,NaN,NaN,NaN
3,Flow,Mihaly Csikszentmihalyi,1990.0,11041932.0,/works/OL278571W,openlibrary,NaN,NaN,NaN,NaN,NaN
4,Psychology,Wayne Weiten,1989.0,363719.0,/works/OL90044W,openlibrary,NaN,NaN,NaN,NaN,NaN


In [110]:
print(df_scrap.shape)
print(df_scrap.head())

(100, 8)
                                   title   price rating_text  \
0                   A Light in the Attic  £51.77       Three   
1                     Tipping the Velvet  £53.74         One   
2                             Soumission  £50.10         One   
3                          Sharp Objects  £47.82        Four   
4  Sapiens: A Brief History of Humankind  £54.23        Five   

                                                link              author  \
0               a-light-in-the-attic_1000/index.html    Shel Silverstein   
1                  tipping-the-velvet_999/index.html        Sarah Waters   
2                          soumission_998/index.html  Michel Houellebecq   
3                       sharp-objects_997/index.html       Gillian Flynn   
4  sapiens-a-brief-history-of-humankind_996/index...          BookNation   

   description  genres    source  
0          NaN     NaN  scraping  
1          NaN     NaN  scraping  
2          NaN     NaN  scraping  
3        

In [111]:
print(api_df.shape)
print(api_df.head())

(500, 5)
                    title                                             author  \
0         Dark Psychology                       Modern Psychology Publishing   
1        Dream Psychology                                      Sigmund Freud   
2  Educational psychology  Anita Woolfolk Hoy, Philip H. Winne, Nancy E. ...   
3                    Flow                            Mihaly Csikszentmihalyi   
4              Psychology                                       Wayne Weiten   

     year    cover_id            work_key  
0  2017.0  13262384.0  /works/OL30150379W  
1  2017.0  13180633.0  /works/OL29743214W  
2  1987.0    137962.0   /works/OL2651233W  
3  1990.0  11041932.0    /works/OL278571W  
4  1989.0    363719.0     /works/OL90044W  


In [112]:
import requests

def enrich(work_key):
    try:
        url = f"https://openlibrary.org{work_key}.json"
        r = requests.get(url, timeout=10)
        data = r.json()

        desc = data.get("description", "")
        if isinstance(desc, dict):
            desc = desc.get("value", "")

        subjects = data.get("subjects", [])

        return desc, subjects
    except:
        return "", []

In [ ]:
descriptions = []
subjects = []

for i, row in df_api.iterrows():
    d, s = enrich(row["work_key"])
    descriptions.append(d)
    subjects.append(", ".join(s) if s else "")

    if i % 50 == 0:
        print("Processed:", i)

Processed: 0
Processed: 50
Processed: 100


In [ ]:
df_api["description"] = descriptions
df_api["subjects"] = subjects

In [ ]:
final_df = pd.concat([df_api, df_scrap], ignore_index=True)

In [ ]:
final_df["content"] = (
    final_df["title"].fillna("") + " " +
    final_df["author"].fillna("") + " " +
    final_df["description"].fillna("") + " " +
    final_df["genres"].fillna("")
)